In [ ]:
import pandas as pd

df = pd.read_csv('Datasets/spam.csv', encoding='latin1')

print("DataFrame Head:")
display(df.head())

print("\nDataFrame Info:")
df.info()

DataFrame Head:


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [2]:
df.rename(columns={'v1':'target', 'v2':'text'}, inplace=True)

df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

print("DataFrame after renaming columns and dropping irrelevant ones:")
display(df.head())

print("\nDataFrame Info after cleaning:")
df.info()

DataFrame after renaming columns and dropping irrelevant ones:


,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



DataFrame Info after cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [3]:
df['target'] = df['target'].map({'ham': 0, 'spam': 1})

print("DataFrame after converting 'target' to numerical labels:")
display(df.head())

print("\nValue counts for the 'target' column:")
print(df['target'].value_counts())

DataFrame after converting 'target' to numerical labels:


,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."



Value counts for the 'target' column:
target
0    4825
1     747
Name: count, dtype: int64


In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download stopwords if not already downloaded
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['text'] = df['text'].apply(preprocess_text)

print("First 5 preprocessed text entries:")
for i, text in enumerate(df['text'].head()):
    print(f"{i}: {text}")

First 5 preprocessed text entries:
0: go jurong point crazi avail bugi n great world la e buffet cine got amor wat
1: ok lar joke wif u oni
2: free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli
3: u dun say earli hor u c alreadi say
4: nah think goe usf live around though


In [6]:
max_words = 10000
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(df['text'])

X = tokenizer.texts_to_sequences(df['text'])

max_len = max(len(seq) for seq in X)
X = pad_sequences(X, maxlen=max_len)

y = df['target'].values

print(f"Max sequence length: {max_len}")
print(f"Shape of preprocessed text data (X): {X.shape}")
print(f"Shape of target labels (y): {y.shape}")

print("\nFirst 5 preprocessed and padded sequences:")
for i in range(5):
    print(f"{i}: {X[i]}")

Max sequence length: 77
Shape of preprocessed text data (X): (5572, 77)
Shape of target labels (y): (5572,)

First 5 preprocessed and padded sequences:
0: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    4 3256  298  601  514 1070   27   69  249
  818   84 2304 1071   19 3257   75]
1: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0   10  260  542  370    2 1543]
2: [   0    0    0    0    0    0    0    0    0    0    0

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (4457, 77)
Shape of X_test: (1115, 77)
Shape of y_train: (4457,)
Shape of y_test: (1115,)


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

embedding_dim = 100

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len))
model.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid')) # Binary classification, so 1 output unit with sigmoid

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("LSTM Model Summary:")
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


LSTM Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

print("\nModel training complete.")

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 42s 280ms/step - accuracy: 0.9346 - loss: 0.1983 - val_accuracy: 0.9742 - val_loss: 0.0749
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 31s 275ms/step - accuracy: 0.9888 - loss: 0.0385 - val_accuracy: 0.9765 - val_loss: 0.0650
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 30s 267ms/step - accuracy: 0.9958 - loss: 0.0175 - val_accuracy: 0.9843 - val_loss: 0.0662
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 41s 266ms/step - accuracy: 0.9986 - loss: 0.0078 - val_accuracy: 0.9809 - val_loss: 0.0664
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 31s 278ms/step - accuracy: 0.9986 - loss: 0.0067 - val_accuracy: 0.9832 - val_loss: 0.0685
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 30s 266ms/step - accuracy: 0.9994 - loss: 0.0045 - val_accuracy: 0.9809 - val_loss: 0.0797
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 31s 279ms/step - accuracy: 0.9994 - loss: 0.0026 - val_accuracy: 0.9809 - val_loss: 0.0837
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 30s 271ms/step - accuracy: 0.9994 - loss: 0

In [10]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.9839 - loss: 0.0531

Test Loss: 0.0531
Test Accuracy: 0.9839


## Project Summary: Spam Detection with LSTM

This project involved building and evaluating an LSTM (Long Short-Term Memory) neural network for spam detection using the `spam.csv` dataset.

### 1. Dataset Overview

The dataset `spam.csv` initially contained 5572 entries and 5 columns. After initial inspection, it was found to contain email/SMS messages labeled as 'ham' (legitimate) or 'spam'. Columns `v1` and `v2` were fully populated, while `Unnamed: 2`, `Unnamed: 3`, and `Unnamed: 4` had a significant number of missing values.

### 2. Data Preprocessing and Feature Selection

*   **Column Renaming:** `v1` was renamed to `target` and `v2` to `text` for better clarity.
*   **Irrelevant Column Removal:** The `Unnamed: 2`, `Unnamed: 3`, and `Unnamed: 4` columns were dropped due to their high cardinality and missing values.
*   **Target Encoding:** The `target` column was converted to numerical labels, with 'ham' mapped to `0` and 'spam' to `1`. The dataset was imbalanced, with 4825 'ham' messages and 747 'spam' messages.
*   **Text Preprocessing:** A `preprocess_text` function was applied to the `text` column, performing the following steps:
    *   Conversion to lowercase.
    *   Removal of non-alphabetic characters.
    *   Removal of English stopwords using NLTK.
    *   Porter Stemming to reduce words to their root form.
*   **Tokenization and Padding:** The preprocessed text was then tokenized using `tensorflow.keras.preprocessing.text.Tokenizer`, considering the top 10,000 most frequent words. The sequences were then padded to a uniform length of 77 (the maximum sequence length observed) to prepare them for the LSTM model. This resulted in `X` having a shape of (5572, 77) and `y` having a shape of (5572,).

### 3. Model Architecture and Training

*   **Data Split:** The preprocessed data was split into training and testing sets with an 80/20 ratio (`random_state=42`), resulting in:
    *   `X_train`: (4457, 77), `X_test`: (1115, 77)
    *   `y_train`: (4457,), `y_test`: (1115,)
*   **LSTM Model:** A Sequential LSTM model was constructed using TensorFlow/Keras:
    *   An `Embedding` layer (input_dim=10000, output_dim=100, input_length=77) to convert integer sequences into dense vectors.
    *   An `LSTM` layer with 128 units, a dropout of 0.2, and recurrent dropout of 0.2 to handle sequence dependencies and prevent overfitting.
    *   A `Dense` output layer with 1 unit and a 'sigmoid' activation function for binary classification.
*   **Compilation:** The model was compiled with the 'adam' optimizer, 'binary_crossentropy' loss function, and 'accuracy' as the metric.
*   **Training:** The model was trained for 10 epochs with a batch size of 32, using 20% of the training data for validation. During training, the model showed significant improvement in accuracy and reduction in loss, achieving a final validation accuracy of approximately 98.21%.

### 4. Model Evaluation and Results

After training, the model was evaluated on the unseen test set:

*   **Test Loss:** 0.0531
*   **Test Accuracy:** 0.9839

The high test accuracy indicates that the LSTM model is performing very well in distinguishing between spam and ham messages on new, unseen data, demonstrating its effectiveness for this classification task.